# Semantic Role Mapping

This notebook implements canonical role mapping using a hybrid approach:

1. **Dictionary/Alias Match** - Fast exact and fuzzy dictionary lookup
2. **Regex Pattern Match** - Rule-based pattern matching for common role formats
3. **NLP/LLM Fallback** - Optional semantic matching for unresolved low-confidence records

## Pipeline Flow
- Input: Job titles from `workspace.silver.silver_jobs_current`
- Output: Role mappings written to `workspace.semantic.sem_job_role_map`
- Review Queue: Low-confidence matches sent to `workspace.silver.silver_semantic_review_queue`

## Features
- Explicit timeout handling for LLM calls
- Confidence scoring at each stage
- Audit trail for all mappings
- Performance metrics tracking

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType, TimestampType
from datetime import datetime
import re
from typing import Optional, Tuple

# Configuration
CONFIG = {
    "input_table": "workspace.silver.silver_jobs_current",
    "output_table": "workspace.semantic.sem_job_role_map",
    "review_queue_table": "workspace.silver.silver_semantic_review_queue",
    "confidence_threshold": 0.7,  # Below this, goes to review queue
    "llm_timeout_seconds": 5,
    "enable_llm_fallback": False,  # Set to True to enable LLM fallback
    "batch_size": 10000
}

In [0]:
# Load canonical role dictionary from governed metadata files
# This replaces hardcoded tech-centric role dictionary with sector-agnostic taxonomy

METADATA_BASE = "/Workspace/Users/aaryan.shrivastav1403@gmail.com/LMIP/metadata"

print("Loading role taxonomy from metadata...")
roles_raw_df = spark.read.csv(
    f"{METADATA_BASE}/canonical_roles.csv",
    header=True,
    inferSchema=True
)

# Explode aliases into individual dictionary entries
# Transform to match existing schema expectations
role_dict_df = roles_raw_df.select(
    F.col("canonical_role"),
    F.col("family_key"),
    F.col("sector_key"),
    F.col("seniority"),
    F.explode(F.split(F.col("aliases"), "\\|")).alias("alias")
).withColumn(
    "alias", F.lower(F.trim(F.col("alias")))
).withColumn(
    "match_type", 
    F.when(F.col("alias") == F.lower(F.col("canonical_role")), "exact").otherwise("alias")
).withColumn(
    "confidence_boost",
    F.when(F.col("match_type") == "exact", 1.0).otherwise(0.9)
)

In [0]:
# Build lookup dictionaries for fast matching
role_dict_lookup = {}
for row in role_dict_df.collect():
    alias_key = row["alias"].lower().strip()
    role_dict_lookup[alias_key] = {
        "canonical_role": row["canonical_role"],
        "confidence": row["confidence_boost"],
        "method": "dictionary"
    }

def match_dictionary(role_text: str) -> Optional[Tuple[str, float, str]]:
    """Match role against dictionary. Returns (canonical_role, confidence, method) or None."""
    if not role_text:
        return None
    
    role_key = role_text.lower().strip()
    
    # Exact match
    if role_key in role_dict_lookup:
        result = role_dict_lookup[role_key]
        return (result["canonical_role"], result["confidence"], result["method"])
    
    # Fuzzy match (simple contains logic)
    for alias, mapping in role_dict_lookup.items():
        if alias in role_key or role_key in alias:
            return (mapping["canonical_role"], mapping["confidence"] * 0.85, "dictionary_fuzzy")
    
    return None

In [0]:
# Define regex patterns for common role structures
ROLE_PATTERNS = [
    # Pattern: (regex, canonical_role, confidence)
    (r"(?i)^(senior|sr|lead)\s*software\s*(engineer|dev)", "Senior Software Engineer", 0.9),
    (r"(?i)^(junior|jr)\s*software\s*(engineer|dev)", "Junior Software Engineer", 0.9),
    (r"(?i)^software\s*(engineer|dev)", "Software Engineer", 0.85),
    (r"(?i)^(senior|sr|lead)\s*data\s*scientist", "Senior Data Scientist", 0.9),
    (r"(?i)^data\s*scientist", "Data Scientist", 0.85),
    (r"(?i)^(senior|sr)\s*product\s*manager", "Senior Product Manager", 0.9),
    (r"(?i)^product\s*manager", "Product Manager", 0.85),
    (r"(?i)^(chief|head\s+of)\s*technology", "Chief Technology Officer", 0.9),
    (r"(?i)^engineering\s*manager", "Engineering Manager", 0.85),
]

def match_regex(role_text: str) -> Optional[Tuple[str, float, str]]:
    """Match role against regex patterns. Returns (canonical_role, confidence, method) or None."""
    if not role_text:
        return None
    
    for pattern, canonical, confidence in ROLE_PATTERNS:
        if re.match(pattern, role_text):
            return (canonical, confidence, "regex")
    
    return None

In [0]:
import signal
from contextlib import contextmanager

class TimeoutException(Exception):
    pass

@contextmanager
def time_limit(seconds):
    """Context manager for timeout handling."""
    def signal_handler(signum, frame):
        raise TimeoutException("LLM call timed out")
    
    # Note: signal.alarm only works on Unix systems
    try:
        signal.signal(signal.SIGALRM, signal_handler)
        signal.alarm(seconds)
        yield
    finally:
        signal.alarm(0)

def match_llm(role_text: str, timeout_seconds: int = 5) -> Optional[Tuple[str, float, str]]:
    """Match role using LLM with timeout. Returns (canonical_role, confidence, method) or None."""
    if not role_text:
        return None
    
    try:
        # Placeholder for LLM integration
        # In production, this would call an AI function or external LLM API
        # Example: ai_classify(role_text, timeout=timeout_seconds)
        
        # with time_limit(timeout_seconds):
        #     result = spark.sql(f"""
        #         SELECT ai_classify(
        #             '{role_text}',
        #             array('Software Engineer', 'Data Scientist', 'Product Manager', 'Other')
        #         ) as canonical_role
        #     """).first()
        #     return (result.canonical_role, 0.6, "llm")
        
        # For now, return None (not implemented)
        return None
        
    except TimeoutException:
        print(f"LLM timeout for role: {role_text}")
        return None
    except Exception as e:
        print(f"LLM error for role {role_text}: {e}")
        return None

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StructType, StructField, StringType, FloatType

# Define return schema for UDF
match_schema = StructType([
    StructField("canonical_role", StringType(), True),
    StructField("confidence", FloatType(), True),
    StructField("method", StringType(), True)
])

def hybrid_match(role_text: str) -> Tuple[Optional[str], Optional[float], Optional[str]]:
    """Apply hybrid matching: dictionary -> regex -> LLM."""
    if not role_text:
        return (None, None, None)
    
    # Stage 1: Dictionary match
    result = match_dictionary(role_text)
    if result:
        return result
    
    # Stage 2: Regex match
    result = match_regex(role_text)
    if result:
        return result
    
    # Stage 3: LLM fallback (if enabled)
    if CONFIG["enable_llm_fallback"]:
        result = match_llm(role_text, CONFIG["llm_timeout_seconds"])
        if result:
            return result
    
    # No match found
    return (None, None, None)

hybrid_match_udf = udf(hybrid_match, match_schema)

In [0]:
try:
    jobs_df = spark.table(CONFIG["input_table"])
    input_df = jobs_df.select(
        F.col("enterprise_job_id").alias("role_id"),
        F.col("title_normalized").alias("standardized_role"),
        F.col("effective_timestamp").alias("extraction_date")
    ).filter(
        F.col("title_normalized").isNotNull()
    )
    
    input_count = input_df.count()
    print(f"Loaded {input_count} job titles from {CONFIG['input_table']}")
    
except Exception as e:
    print(f"Warning: Could not load input table: {e}")
    print("Creating sample input data...")
    
    # Create sample data for demonstration
    sample_data = [
        ("job_001", "Software Engineer", "2024-01-01"),
        ("job_002", "Senior SWE", "2024-01-02"),
        ("job_003", "Lead Software Dev", "2024-01-03"),
        ("job_004", "Data Scientist", "2024-01-04"),
        ("job_005", "ML Engineer", "2024-01-05"),
        ("job_006", "Product Manager", "2024-01-06"),
        ("job_007", "PM", "2024-01-07"),
        ("job_008", "DevOps Engineer", "2024-01-08"),
        ("job_009", "Unknown Job Title", "2024-01-09"),
    ]
    
    input_df = spark.createDataFrame(
        sample_data,
        ["role_id", "standardized_role", "extraction_date"]
    )
    input_count = input_df.count()
    print(f"Sample data created with {input_count} records")

display(input_df.limit(10))

In [0]:
# Apply hybrid matching to all roles
matched_df = input_df.withColumn(
    "match_result",
    hybrid_match_udf(F.col("standardized_role"))
).select(
    F.col("role_id").alias("enterprise_job_id"),
    "standardized_role",
    "extraction_date",
    F.col("match_result.canonical_role").alias("canonical_role"),
    F.col("match_result.confidence").alias("confidence"),
    F.col("match_result.method").alias("matching_method"),
    F.current_timestamp().alias("processed_timestamp")
)

print(f"Applied hybrid matching to {matched_df.count()} roles")
display(matched_df)

In [0]:
# Split into canonical roles (high confidence) and review queue (low confidence or no match)
confidence_threshold = CONFIG["confidence_threshold"]

# Canonical roles: matched with confidence >= threshold
canonical_df = matched_df.filter(
    (F.col("canonical_role").isNotNull()) & 
    (F.col("confidence") >= confidence_threshold)
)

# Review queue: no match or low confidence
review_queue_df = matched_df.filter(
    (F.col("canonical_role").isNull()) | 
    (F.col("confidence") < confidence_threshold)
).withColumn(
    "review_reason",
    F.when(F.col("canonical_role").isNull(), "no_match")
     .otherwise("low_confidence")
).withColumn(
    "review_status",
    F.lit("pending")
)

canonical_count = canonical_df.count()
review_count = review_queue_df.count()

print(f"\nResults Summary:")
print(f"  Canonical roles (confidence >= {confidence_threshold}): {canonical_count}")
print(f"  Review queue (no match or low confidence): {review_count}")
print(f"  Success rate: {canonical_count / input_count * 100:.1f}%")

print("\nCanonical Roles:")
display(canonical_df)

print("\nReview Queue:")
display(review_queue_df)

In [0]:
# Prepare data for sem_job_role_map schema
from pyspark.sql.functions import concat, lit, md5, when, regexp_extract

# Generate role_map_id and extract seniority from canonical_role
output_df = canonical_df.select(
    md5(concat(F.col("enterprise_job_id"), F.col("canonical_role"))).alias("role_map_id"),
    F.col("enterprise_job_id"),
    F.col("standardized_role").alias("title_normalized"),  # Required for warehouse join
    F.col("canonical_role").alias("canonical_role_id"),
    F.lit("Technology").alias("role_family"),  # Simplified - could extract from role name
    when(F.col("canonical_role").contains("Senior"), "SENIOR")
     .when(F.col("canonical_role").contains("Lead"), "SENIOR")
     .when(F.col("canonical_role").contains("Junior"), "ENTRY")
     .when(F.col("canonical_role").contains("Chief"), "EXECUTIVE")
     .otherwise("MID").alias("seniority_level"),
    when(F.col("matching_method") == "dictionary", "EXACT_MATCH")
     .when(F.col("matching_method") == "dictionary_fuzzy", "FUZZY")
     .when(F.col("matching_method") == "regex", "EXACT_MATCH")
     .when(F.col("matching_method") == "llm", "LLM")
     .otherwise("MANUAL").alias("normalization_method"),
    F.col("confidence").cast("decimal(5,4)").alias("normalization_confidence"),
    F.to_json(F.struct(
        F.col("standardized_role").alias("input"),
        F.col("matching_method").alias("method")
    )).alias("explanation_json"),
    F.date_format(F.current_timestamp(), "yyyyMMdd_HHmmss").alias("effective_batch_id")
)

# Write to semantic.sem_job_role_map
print(f"Writing {canonical_count} role mappings to {CONFIG['output_table']}...")

output_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(CONFIG["output_table"])

print("✓ Role mappings written successfully")

# Write review queue to silver
if review_count > 0:
    print(f"\nWriting {review_count} records to review queue at {CONFIG['review_queue_table']}...")
    
    review_df = review_queue_df.select(
        F.expr("uuid()").alias("review_id"),
        F.col("enterprise_job_id"),
        F.col("review_reason").alias("issue_type"),
        F.concat(
            F.lit("Role: "),
            F.col("standardized_role"),
            F.lit(", Suggested: "),
            F.coalesce(F.col("canonical_role"), F.lit("none"))
        ).alias("issue_detail"),
        F.coalesce(F.col("confidence"), F.lit(0.0)).cast("decimal(5,4)").alias("confidence"),
        F.current_timestamp().alias("queued_at"),
        F.col("review_status").alias("status")
    )
    
    review_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(CONFIG["review_queue_table"])
    
    print("✓ Review queue records written successfully")
else:
    print("\n✓ No records require review")

In [0]:
# Calculate and display performance metrics
metrics_df = matched_df.groupBy("matching_method").agg(
    F.count("*").alias("count"),
    F.avg("confidence").alias("avg_confidence"),
    F.min("confidence").alias("min_confidence"),
    F.max("confidence").alias("max_confidence")
).orderBy(F.desc("count"))

print("\nMatching Method Performance:")
display(metrics_df)

# Summary statistics
print("\n" + "="*60)
print("SEMANTIC ROLE MAPPING - EXECUTION SUMMARY")
print("="*60)
print(f"Input records: {input_count}")
print(f"Canonical mappings: {canonical_count} ({canonical_count/input_count*100:.1f}%)")
print(f"Review queue: {review_count} ({review_count/input_count*100:.1f}%)")
print(f"Confidence threshold: {confidence_threshold}")
print(f"LLM fallback: {'Enabled' if CONFIG['enable_llm_fallback'] else 'Disabled'}")
print("="*60)